In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [2]:
production = pd.read_csv(
    "../datasets/processed/production_features.csv"
)

print(production.shape)

production.head()

(233715, 30)


,State,District_Name,Year,Season,Crop,Area,Production,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
0,Andaman And Nicobar Islands,South Andamans,2002,Whole Year,Arecanut,3105.00,5267.00,27.209282,23.616293,60.019684,...,0.535426,-0.922032,-0.107663,5,25,1633.092508,642.582376,265.000949,482.623741,218.792313
1,Andaman And Nicobar Islands,Nicobars,2003,Whole Year,Arecanut,1261.00,1525.00,27.149813,23.356646,65.769770,...,0.533881,0.927362,0.121350,6,36,1785.636957,634.128571,296.733932,469.625954,193.861965
2,Andaman And Nicobar Islands,South Andamans,2003,Whole Year,Arecanut,3118.00,5182.00,27.149813,23.356646,65.769770,...,0.644495,-1.239001,-0.144584,6,36,1785.636957,634.128571,296.733932,529.135407,218.427514
3,Andaman And Nicobar Islands,Nicobars,2004,Whole Year,Arecanut,1264.70,805.85,27.089384,23.254425,66.004180,...,0.710688,1.222734,0.166803,7,49,1788.012578,629.948049,289.209822,471.492979,193.509780
4,Andaman And Nicobar Islands,South Andamans,2004,Whole Year,Arecanut,3160.67,3975.20,27.089384,23.254425,66.004180,...,0.945233,-1.860002,-0.217464,7,49,1788.012578,629.948049,289.209822,531.918159,218.309435


In [3]:
production.info()

<class 'pandas.DataFrame'>
RangeIndex: 233715 entries, 0 to 233714
Data columns (total 30 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   State                        233715 non-null  str    
 1   District_Name                233715 non-null  str    
 2   Year                         233715 non-null  int64  
 3   Season                       233715 non-null  str    
 4   Crop                         233715 non-null  str    
 5   Area                         233715 non-null  float64
 6   Production                   233715 non-null  float64
 7   Avg_Annual_Temperature       233715 non-null  float64
 8   Avg_Annual_DewPoint          233715 non-null  float64
 9   Total_Annual_Rainfall        233715 non-null  float64
 10  Avg_Annual_Wind_Speed        233715 non-null  float64
 11  Avg_Annual_Surface_Pressure  233715 non-null  float64
 12  Avg_Grid_Points              233715 non-null  float64
 13  Log_Produc

In [4]:
train = production[production["Year"] <= 2013].copy()
test = production[production["Year"] >= 2014].copy()

print(train.shape)
print(test.shape)

print(train["Year"].min(), "-", train["Year"].max())
print(test["Year"].min(), "-", test["Year"].max())

(222383, 30)
(11332, 30)
1997 - 2013
2014 - 2015


In [5]:
target = "Log_Production"

drop_cols = [
    "Production",
    "Log_Production"
]

X_train = train.drop(columns=drop_cols)

y_train = train[target]

X_test = test.drop(columns=drop_cols)

y_test = test[target]

In [6]:
categorical_features = [
    "State",
    "District_Name",
    "Season",
    "Crop"
]

numeric_features = [
    col
    for col in X_train.columns
    if col not in categorical_features
]

print("Categorical:", categorical_features)
print("Numeric:", len(numeric_features))

Categorical: ['State', 'District_Name', 'Season', 'Crop']
Numeric: 24


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numeric_features
        )
    ]
)

In [8]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [19]:
import numpy as np

numeric_cols = production.select_dtypes(include=np.number).columns

print("Positive Infinity:")
print(np.isposinf(production[numeric_cols]).sum().sort_values(ascending=False).head(20))

print("\nNegative Infinity:")
print(np.isneginf(production[numeric_cols]).sum().sort_values(ascending=False).head(20))

Positive Infinity:
Year                           0
Area                           0
Production                     0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points                0
Log_Production                 0
Log_Area                       0
Production_Lag_1               0
Production_Lag_2               0
Production_Lag_3               0
Rolling_Mean_3                 0
Rolling_Mean_5                 0
Rolling_STD_3                  0
Previous_Change                0
Previous_Growth                0
Years_From_Start               0
dtype: int64

Negative Infinity:
Year                           0
Area                           0
Production                     0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points         

In [16]:
import numpy as np

production.replace([np.inf, -np.inf], np.nan, inplace=True)

,State,District_Name,Year,Season,Crop,Area,Production,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
0,Andaman And Nicobar Islands,South Andamans,2002,Whole Year,Arecanut,3105.00,5267.00,27.209282,23.616293,60.019684,...,0.535426,-0.922032,-0.107663,5,25,1633.092508,642.582376,265.000949,482.623741,218.792313
1,Andaman And Nicobar Islands,Nicobars,2003,Whole Year,Arecanut,1261.00,1525.00,27.149813,23.356646,65.769770,...,0.533881,0.927362,0.121350,6,36,1785.636957,634.128571,296.733932,469.625954,193.861965
2,Andaman And Nicobar Islands,South Andamans,2003,Whole Year,Arecanut,3118.00,5182.00,27.149813,23.356646,65.769770,...,0.644495,-1.239001,-0.144584,6,36,1785.636957,634.128571,296.733932,529.135407,218.427514
3,Andaman And Nicobar Islands,Nicobars,2004,Whole Year,Arecanut,1264.70,805.85,27.089384,23.254425,66.004180,...,0.710688,1.222734,0.166803,7,49,1788.012578,629.948049,289.209822,471.492979,193.509780
4,Andaman And Nicobar Islands,South Andamans,2004,Whole Year,Arecanut,3160.67,3975.20,27.089384,23.254425,66.004180,...,0.945233,-1.860002,-0.217464,7,49,1788.012578,629.948049,289.209822,531.918159,218.309435
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
233710,West Bengal,Medinipur East,2014,Rabi,Wheat,524.00,1444.00,25.482965,20.009310,53.839790,...,3.241846,1.204629,0.114302,17,289,1371.997484,509.896546,72.497140,337.220047,159.609959
233711,West Bengal,Medinipur West,2014,Rabi,Wheat,5769.00,12259.00,25.482965,20.009310,53.839790,...,2.311557,-4.467744,-0.380440,17,289,1371.997484,509.896546,72.497140,466.275590,220.693367
233712,West Bengal,Murshidabad,2014,Rabi,Wheat,104907.00,300497.00,25.482965,20.009310,53.839790,...,2.234554,2.138233,0.293880,17,289,1371.997484,509.896546,72.497140,622.433147,294.604457
233713,West Bengal,Nadia,2014,Rabi,Wheat,43551.00,148327.00,25.482965,20.009310,53.839790,...,2.686180,3.199099,0.339820,17,289,1371.997484,509.896546,72.497140,575.101072,272.201665


In [17]:
np.isinf(production.select_dtypes(include=np.number)).sum().sum()

np.int64(0)

In [18]:
production.dropna(inplace=True)

print(production.shape)

(233175, 30)


In [21]:
production[numeric_cols].max().sort_values(ascending=False).head(20)

Production                     1.250800e+09
Area                           8.580100e+06
Temp_Rainfall                  3.271797e+03
Year                           2.015000e+03
Area_Rainfall                  1.781888e+03
Avg_Annual_Surface_Pressure    1.007696e+03
Previous_Growth                1.006080e+03
Temp_DewPoint                  6.626167e+02
Avg_Grid_Points                4.970000e+02
Area_Temperature               3.984248e+02
Rainfall_Wind                  3.508712e+02
Year_Squared                   3.240000e+02
Total_Annual_Rainfall          1.758574e+02
Avg_Annual_Temperature         2.833691e+01
Avg_Annual_DewPoint            2.376511e+01
Log_Production                 2.094705e+01
Production_Lag_2               2.094705e+01
Production_Lag_1               2.094705e+01
Production_Lag_3               2.094705e+01
Rolling_Mean_3                 2.046255e+01
dtype: float64

In [15]:
inf_rows = production[
    np.isinf(production[numeric_cols]).any(axis=1)
]

print(inf_rows.shape)
inf_rows.head()

(540, 30)


,State,District_Name,Year,Season,Crop,Area,Production,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
1083,Andhra Pradesh,Chittoor,2012,Rabi,Cabbage,179.0,2027.0,27.572472,19.865290,28.807112,...,4.093939,7.090910,inf,15,225,794.283289,547.735152,59.686098,149.594090,143.182657
4059,Andhra Pradesh,Chittoor,2004,Rabi,Linseed,11.0,1.0,27.042295,20.051485,29.056385,...,0.986064,1.609438,inf,7,49,785.751335,542.238173,59.004351,72.202404,67.197579
5527,Andhra Pradesh,Prakasam,2000,Rabi,Other Rabi Pulses,134.0,66.0,26.960619,20.307186,32.391197,...,2.829248,2.302585,inf,3,9,873.286721,547.494305,64.852113,158.887722,132.249244
17306,Assam,Jorhat,2003,Kharif,Mesta,2.0,10.0,23.217352,19.622389,93.987550,...,1.624139,3.135494,inf,6,36,2182.142032,455.579912,54.555666,103.255877,25.506868
21410,Assam,Darrang,2003,Autumn,Rice,58340.0,60241.0,23.217352,19.622389,93.987550,...,6.502168,12.282297,inf,6,36,2182.142032,455.579912,54.555666,1031.425049,254.788623


In [23]:
import numpy as np

numeric_cols = X_train.select_dtypes(include=np.number).columns

print("NaN values:")
print(X_train[numeric_cols].isna().sum().sort_values(ascending=False).head(20))

print("\nPositive Inf:")
print(np.isposinf(X_train[numeric_cols]).sum().sort_values(ascending=False).head(20))

print("\nNegative Inf:")
print(np.isneginf(X_train[numeric_cols]).sum().sort_values(ascending=False).head(20))

NaN values:
Year                           0
Area                           0
Avg_Annual_Temperature         0
Avg_Annual_DewPoint            0
Total_Annual_Rainfall          0
Avg_Annual_Wind_Speed          0
Avg_Annual_Surface_Pressure    0
Avg_Grid_Points                0
Log_Area                       0
Production_Lag_1               0
Production_Lag_2               0
Production_Lag_3               0
Rolling_Mean_3                 0
Rolling_Mean_5                 0
Rolling_STD_3                  0
Previous_Change                0
Previous_Growth                0
Years_From_Start               0
Year_Squared                   0
Temp_Rainfall                  0
dtype: int64

Positive Inf:
Previous_Growth                539
Year                             0
Avg_Annual_Temperature           0
Area                             0
Total_Annual_Rainfall            0
Avg_Annual_Wind_Speed            0
Avg_Annual_Surface_Pressure      0
Avg_Annual_DewPoint              0
Avg_Grid_Points     

In [24]:
X_train[numeric_cols].max().sort_values(ascending=False).head(30)

Previous_Growth                         inf
Area                           8.580100e+06
Temp_Rainfall                  3.271797e+03
Year                           2.013000e+03
Area_Rainfall                  1.781888e+03
Avg_Annual_Surface_Pressure    1.007696e+03
Temp_DewPoint                  6.626167e+02
Avg_Grid_Points                4.970000e+02
Area_Temperature               3.984248e+02
Rainfall_Wind                  3.508712e+02
Year_Squared                   2.560000e+02
Total_Annual_Rainfall          1.758574e+02
Avg_Annual_Temperature         2.833691e+01
Avg_Annual_DewPoint            2.376511e+01
Production_Lag_2               2.094705e+01
Production_Lag_1               2.094705e+01
Production_Lag_3               2.094705e+01
Rolling_Mean_3                 2.046255e+01
Rolling_Mean_5                 2.005075e+01
Years_From_Start               1.600000e+01
Log_Area                       1.596496e+01
Previous_Change                1.379531e+01
Rolling_STD_3                  8

In [25]:
X_train_processed = preprocessor.fit_transform(X_train)

print(type(X_train_processed))
print(X_train_processed.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(222383, 813)


In [28]:
print(type(X_train_processed))

<class 'scipy.sparse._csr.csr_matrix'>


In [29]:
X_dense = X_train_processed.toarray()

print("NaN:", np.isnan(X_dense).sum())
print("Inf:", np.isinf(X_dense).sum())

NaN: 0
Inf: 539


In [30]:
print("NaN:", np.isnan(X_train_processed.data).sum())
print("Inf:", np.isinf(X_train_processed.data).sum())
print("Max:", X_train_processed.data.max())
print("Min:", X_train_processed.data.min())

NaN: 0
Inf: 539
Max: inf
Min: -13.981025843899117


In [31]:
print(np.isinf(production.select_dtypes(include=np.number)).sum().sum())

0


In [32]:
print(production.isna().sum().sum())

0


In [33]:
print(np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

539


In [34]:
print(X_train.select_dtypes(include=np.number).max())

Year                           2.013000e+03
Area                           8.580100e+06
Avg_Annual_Temperature         2.833691e+01
Avg_Annual_DewPoint            2.376511e+01
Total_Annual_Rainfall          1.758574e+02
Avg_Annual_Wind_Speed          4.511707e+00
Avg_Annual_Surface_Pressure    1.007696e+03
Avg_Grid_Points                4.970000e+02
Log_Area                       1.596496e+01
Production_Lag_1               2.094705e+01
Production_Lag_2               2.094705e+01
Production_Lag_3               2.094705e+01
Rolling_Mean_3                 2.046255e+01
Rolling_Mean_5                 2.005075e+01
Rolling_STD_3                  8.021091e+00
Previous_Change                1.379531e+01
Previous_Growth                         inf
Years_From_Start               1.600000e+01
Year_Squared                   2.560000e+02
Temp_Rainfall                  3.271797e+03
Temp_DewPoint                  6.626167e+02
Rainfall_Wind                  3.508712e+02
Area_Rainfall                  1

In [35]:
X_train.dtypes

State                              str
District_Name                      str
Year                             int64
Season                             str
Crop                               str
Area                           float64
Avg_Annual_Temperature         float64
Avg_Annual_DewPoint            float64
Total_Annual_Rainfall          float64
Avg_Annual_Wind_Speed          float64
Avg_Annual_Surface_Pressure    float64
Avg_Grid_Points                float64
Log_Area                       float64
Production_Lag_1               float64
Production_Lag_2               float64
Production_Lag_3               float64
Rolling_Mean_3                 float64
Rolling_Mean_5                 float64
Rolling_STD_3                  float64
Previous_Change                float64
Previous_Growth                float64
Years_From_Start                 int64
Year_Squared                     int64
Temp_Rainfall                  float64
Temp_DewPoint                  float64
Rainfall_Wind            

In [36]:
X_train.head()

,State,District_Name,Year,Season,Crop,Area,Avg_Annual_Temperature,Avg_Annual_DewPoint,Total_Annual_Rainfall,Avg_Annual_Wind_Speed,...,Rolling_STD_3,Previous_Change,Previous_Growth,Years_From_Start,Year_Squared,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind,Area_Rainfall,Area_Temperature
0,Andaman And Nicobar Islands,South Andamans,2002,Whole Year,Arecanut,3105.00,27.209282,23.616293,60.019684,4.415234,...,0.535426,-0.922032,-0.107663,5,25,1633.092508,642.582376,265.000949,482.623741,218.792313
1,Andaman And Nicobar Islands,Nicobars,2003,Whole Year,Arecanut,1261.00,27.149813,23.356646,65.769770,4.511707,...,0.533881,0.927362,0.121350,6,36,1785.636957,634.128571,296.733932,469.625954,193.861965
2,Andaman And Nicobar Islands,South Andamans,2003,Whole Year,Arecanut,3118.00,27.149813,23.356646,65.769770,4.511707,...,0.644495,-1.239001,-0.144584,6,36,1785.636957,634.128571,296.733932,529.135407,218.427514
3,Andaman And Nicobar Islands,Nicobars,2004,Whole Year,Arecanut,1264.70,27.089384,23.254425,66.004180,4.381690,...,0.710688,1.222734,0.166803,7,49,1788.012578,629.948049,289.209822,471.492979,193.509780
4,Andaman And Nicobar Islands,South Andamans,2004,Whole Year,Arecanut,3160.67,27.089384,23.254425,66.004180,4.381690,...,0.945233,-1.860002,-0.217464,7,49,1788.012578,629.948049,289.209822,531.918159,218.309435


In [37]:
# Recreate train/test
train = production[production["Year"] <= 2013].copy()
test = production[production["Year"] >= 2014].copy()

# Recreate X and y
target = "Log_Production"

drop_cols = [
    "Production",
    "Log_Production"
]

X_train = train.drop(columns=drop_cols)
y_train = train[target]

X_test = test.drop(columns=drop_cols)
y_test = test[target]

In [38]:
print(np.isinf(X_train.select_dtypes(include=np.number)).sum().sum())

0


In [41]:
drop_cols = [
    "Production",
    "Log_Production",
    "Area",              # keep Log_Area instead
    "Years_From_Start",
    "Year_Squared"
]

In [42]:
print(production["Crop"].nunique())
print(production["District_Name"].nunique())
print(production["State"].nunique())
print(production["Season"].nunique())

105
646
33
6


In [43]:
import time

start = time.time()

rf_pipeline.fit(X_train, y_train)

print("Training Time:", time.time() - start)

KeyboardInterrupt: 

In [44]:
import time

start = time.time()

X_train_processed = preprocessor.fit_transform(X_train)

print("Preprocessing time:", time.time() - start)
print(X_train_processed.shape)

Preprocessing time: 3.1480844020843506
(221844, 813)


In [46]:
from sklearn.ensemble import RandomForestRegressor
import time

rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

start = time.time()

rf.fit(X_train_processed, y_train)

print("Time:", time.time() - start)

KeyboardInterrupt: 

In [40]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                max_depth=20,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

In [39]:
rf_pipeline.fit(X_train, y_train)

KeyboardInterrupt: 

### XG Boost

In [47]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
from xgboost import XGBRegressor

In [49]:
xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),

        (
            "model",
            XGBRegressor(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=8,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                objective="reg:squarederror",
                tree_method="hist",
                n_jobs=-1
            )
        )
    ]
)

In [50]:
xgb_pipeline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [51]:
pred_log = xgb_pipeline.predict(X_test)

pred_original = np.expm1(pred_log)

actual_original = np.expm1(y_test)

In [52]:
log_mae = mean_absolute_error(
    y_test,
    pred_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_log
    )
)

log_r2 = r2_score(
    y_test,
    pred_log
)

print("===== LOG SCALE =====")
print("MAE :", log_mae)
print("RMSE :", log_rmse)
print("R2 :", log_r2)

===== LOG SCALE =====
MAE : 0.32032394920809004
RMSE : 0.4525586627045794
R2 : 0.9793719281666178


In [53]:
mae = mean_absolute_error(
    actual_original,
    pred_original
)

rmse = np.sqrt(
    mean_squared_error(
        actual_original,
        pred_original
    )
)

r2 = r2_score(
    actual_original,
    pred_original
)

print("\n===== ORIGINAL SCALE =====")
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 :", r2)


===== ORIGINAL SCALE =====
MAE : 222630.36479017086
RMSE : 7044383.028893195
R2 : 0.8889750639040483


In [54]:
feature_names = (
    xgb_pipeline.named_steps["preprocessor"]
    .get_feature_names_out()
)

importance = (
    xgb_pipeline.named_steps["model"]
    .feature_importances_
)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)

,Feature,Importance
797,num__Log_Area,0.164212
790,num__Area,0.121552
682,cat__Season_Whole Year,0.031462
776,cat__Crop_Sugarcane,0.030060
812,num__Area_Temperature,0.028867
802,num__Rolling_Mean_5,0.026595
744,cat__Crop_Onion,0.024444
760,cat__Crop_Potato,0.020274
778,cat__Crop_Sweet Potato,0.019598
708,cat__Crop_Coconut,0.014992


In [55]:
joblib.dump(
    xgb_pipeline,
    "../models/xgboost_production.pkl"
)

['../models/xgboost_production.pkl']

### CatBoostRegressor


In [56]:
from catboost import CatBoostRegressor

In [57]:
categorical_features = [
    "State",
    "District_Name",
    "Season",
    "Crop"
]

In [58]:
target = "Log_Production"

drop_cols = [
    "Production",
    "Log_Production"
]

X_train = train.drop(columns=drop_cols)
y_train = train[target]

X_test = test.drop(columns=drop_cols)
y_test = test[target]

In [59]:
cat_features = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

In [60]:
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

In [61]:
cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_features
)

0:	learn: 3.0193326	total: 390ms	remaining: 3m 14s
100:	learn: 0.6418462	total: 15.8s	remaining: 1m 2s
200:	learn: 0.5615960	total: 31.6s	remaining: 47s
300:	learn: 0.5208023	total: 48.7s	remaining: 32.2s
400:	learn: 0.4961892	total: 1m 5s	remaining: 16.1s
499:	learn: 0.4807442	total: 1m 21s	remaining: 0us


CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

In [62]:
pred_log = cat_model.predict(X_test)

pred_original = np.expm1(pred_log)

actual_original = np.expm1(y_test)

In [65]:
log_mae = mean_absolute_error(
    y_test,
    pred_log
)

log_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        pred_log
    )
)

log_r2 = r2_score(
    y_test,
    pred_log
)

print("===== LOG SCALE =====")
print("MAE :", log_mae)
print("RMSE :", log_rmse)
print("R2 :", log_r2)

===== LOG SCALE =====
MAE : 0.32938039750275244
RMSE : 0.4655093432688563
R2 : 0.9781744259817704


In [66]:
mae = mean_absolute_error(
    actual_original,
    pred_original
)

rmse = np.sqrt(
    mean_squared_error(
        actual_original,
        pred_original
    )
)

r2 = r2_score(
    actual_original,
    pred_original
)

print("\n===== ORIGINAL SCALE =====")
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 :", r2)


===== ORIGINAL SCALE =====
MAE : 223043.63006237126
RMSE : 6033325.895210246
R2 : 0.9185580474707669


In [63]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": cat_model.get_feature_importance()
})

feature_importance.sort_values(
    "Importance",
    ascending=False
).head(20)

,Feature,Importance
12,Log_Area,29.983670
5,Area,20.234961
4,Crop,19.455355
17,Rolling_Mean_5,7.707991
3,Season,4.746136
16,Rolling_Mean_3,4.017524
27,Area_Temperature,3.607082
0,State,1.786115
10,Avg_Annual_Surface_Pressure,1.246071
15,Production_Lag_3,1.216708


In [64]:
joblib.dump(
    cat_model,
    "../models/catboost_production.pkl"
)


['../models/catboost_production.pkl']

In [67]:
comparison = pd.DataFrame({
    "Model": ["XGBoost", "CatBoost"],
    "Log_MAE": [0.320324, 0.329380],
    "Log_RMSE": [0.452559, 0.465509],
    "Log_R2": [0.979372, 0.978174],
    "Original_MAE": [222630.36, 223043.63],
    "Original_RMSE": [7044383.03, 6033325.90],
    "Original_R2": [0.888975, 0.918558]
})

comparison

,Model,Log_MAE,Log_RMSE,Log_R2,Original_MAE,Original_RMSE,Original_R2
0,XGBoost,0.320324,0.452559,0.979372,222630.36,7044383.03,0.888975
1,CatBoost,0.329380,0.465509,0.978174,223043.63,6033325.90,0.918558


In [69]:
comparison.to_csv(
    "../reports/production_model_comparison.csv",
    index=False
)